In [ ]:
from ruamel import yaml

# load YAML analysis "manifest", i.e. list of samples to include 
# plus info about metadata to include from acquisition_info files
with open('jurkat-nucleolin_analysis_manifest.yaml') as fd:
    YAML = yaml.YAML()
    dataset_info = YAML.load(fd)

# append all sample infos to a dict of default parameters (e.g. subdirs that are always the same
for sample in dataset_info['samples']:
    sample_plusdefaults = dataset_info['defaults'].copy()
    sample_plusdefaults.update(sample)
    sample.update(sample_plusdefaults)

dataset_info

In [ ]:
from pathlib import Path
from calmutils.misc.json import query_json
import pandas as pd
import os

dfs_all = []

for sample in dataset_info['samples']:
    path = sample['in_path']   
    acqusition_info_path = next(Path(path).glob('*acquisition[-|_]info*.json'))
    print(acqusition_info_path)

    csv_files = sorted((Path(path) / sample['distances_path']).glob('[!.]*.csv'))
    
    dfs_persample = []
    for csv_file in csv_files:
        dfi = pd.read_csv(csv_file)
        
        # add distance file, make image path relative, add base_path as extra column
        dfi['distance_file'] = os.path.relpath(csv_file, sample['in_path'])
        dfi['image_file'] = dfi['image_file'].apply(lambda p: os.path.relpath(p, sample['in_path']))
        dfi['base_path'] = sample['in_path']
        
        dfs_persample.append(dfi)
        
    df = pd.concat(dfs_persample).reset_index(drop=True)

    # query infos from 
    for query in dataset_info['acquisition_info_queries']:
        res = query_json(acqusition_info_path, query)
        df[query] = res

    dfs_all.append(df)


In [ ]:
df = pd.concat(dfs_all)

# drop spots outside of nucleus
df=df[df['label_nucleus'] != 0]

# simpler FISH target (drop probe number)
df['fish_target'] = df['preparation/targets[2]'].str.split(' ', expand=True)[0]

# only keep 561nm spots, i.e. drop all spots from 640nm channel
df = df[df['channel'] == '561-CSU-W1']

# common naming for WT samples
df['experiment/condition'] = df['experiment/condition'].replace({'WT': 'WT 0h'})

df = df.reset_index(drop=True)
df

In [ ]:
import seaborn as sns
sns.set()

# plot for norm. dist. to nuclear border
g = sns.FacetGrid(df, col='fish_target', hue='experiment/condition', height=5)
g.map_dataframe(sns.boxplot, y='q_nucleus', x='experiment/condition')

# norm. dist to closest nuleolus
g = sns.FacetGrid(df, col='fish_target', hue='experiment/condition', height=5)
g.map_dataframe(sns.boxplot, y='q_nucleolus', x='experiment/condition')